<a href="https://colab.research.google.com/github/ABugDrone/Machine-Learning-Projects/blob/main/Building_a_Real_Time_Object_Detection_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Set Up Google Colab Environment

In [ ]:
# Install required libraries
!pip install torch torchvision torchaudio
!pip install kaggle
!pip install kagglehub


Step 2: Import and Download the Dataset

You can use the Kaggle dataset code you provided to download the Pascal VOC dataset.

In [ ]:
import kagglehub

# Download latest version of Pascal VOC 2012 dataset
path = kagglehub.dataset_download("gopalbhattrai/pascal-voc-2012-dataset")
print("Path to dataset files:", path)


100%|██████████| 3.52G/3.52G [00:37<00:00, 101MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/gopalbhattrai/pascal-voc-2012-dataset/versions/1


Step 3: Import Libraries for Object Detection.

In [ ]:
import torch
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
import os


Step 4: Prepare the Dataset

Pascal VOC images typically come in XML format, describing the objects in the image. You may need to convert the annotations into a format compatible with PyTorch’s DataLoader for training.

In [ ]:
from xml.etree import ElementTree as ET
import os
from torchvision.datasets import VOCDetection

# Function to load Pascal VOC annotations
def load_voc_annotations(path):
    # Load the images and their corresponding annotations
    data = VOCDetection(path, year='2012', image_set='trainval', download=True)
    return data

# Load dataset
voc_data = load_voc_annotations(path)


100%|██████████| 2.00G/2.00G [00:42<00:00, 46.8MB/s]


Extracting /root/.cache/kagglehub/datasets/gopalbhattrai/pascal-voc-2012-dataset/versions/1/VOCtrainval_11-May-2012.tar to /root/.cache/kagglehub/datasets/gopalbhattrai/pascal-voc-2012-dataset/versions/1


Step 5: Preprocess and Augment the Data

You can apply common preprocessing techniques and augmentations (e.g., scaling, flipping) to enhance the model's ability to generalize.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((600, 600)),  # Resize images to a consistent size
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Apply transformations to the dataset
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((600, 600)),
    transforms.RandomHorizontalFlip(),  # Augment the data with random horizontal flips
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Split the dataset into train and validation sets
train_data, val_data = train_test_split(voc_data, test_size=0.2, random_state=42)


Step 6: Load the Pre-trained Faster R-CNN Model

PyTorch's torchvision library provides a pre-trained Faster R-CNN model, which we can fine-tune.

In [ ]:
# Load pre-trained Faster R-CNN model with a ResNet backbone
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

# Change the number of output classes to match Pascal VOC (21 classes including background)
num_classes = 21  # 20 classes + background
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)

# Move the model to GPU if available
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:01<00:00, 140MB/s]


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(


Step 7: Define the Training and Evaluation Loops

Now, set up the training loop, evaluation metrics, and optimization.

In [ ]:
# Define the training loop
def train_model(model, train_data, val_data, num_epochs=10, lr=0.005):
    # Prepare DataLoader
    train_loader = DataLoader(train_data, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    val_loader = DataLoader(val_data, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

    # Use Adam optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Define loss functions (the Faster R-CNN model already has built-in losses)
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for images, targets in train_loader:
            images = [image.to(device) for image in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Zero the gradients
            optimizer.zero_grad()

            # Forward pass
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            # Backward pass
            losses.backward()
            optimizer.step()

            running_loss += losses.item()

        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader)}')

    return model


Step 8: Evaluate the Model's Performance

You can evaluate the model using Mean Average Precision (mAP) and Intersection over Union (IoU) metrics.

In [ ]:
from torchvision.models.detection import FasterRCNN
from sklearn.metrics import average_precision_score

# Evaluation function
def evaluate_model(model, val_data):
    model.eval()
    all_boxes = []
    all_labels = []
    all_scores = []

    val_loader = DataLoader(val_data, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

    for images, targets in val_loader:
        images = [image.to(device) for image in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        with torch.no_grad():
            predictions = model(images)

        for i, prediction in enumerate(predictions):
            boxes = prediction['boxes'].cpu().numpy()
            labels = prediction['labels'].cpu().numpy()
            scores = prediction['scores'].cpu().numpy()

            all_boxes.append(boxes)
            all_labels.append(labels)
            all_scores.append(scores)

    # Calculate mAP (Mean Average Precision) and IoU (Intersection over Union)
    # Implement IoU and mAP calculation
    # Example: use scikit-learn or custom metric calculation for evaluation


Step 9: Fine-tuning and Hyperparameter Tuning

1. Learning Rate Adjustment

The learning rate is one of the most critical hyperparameters. If it’s too high, the model may converge too quickly to a suboptimal solution or even fail to converge. If it’s too low, the training process may be too slow.

You can experiment with different learning rates using a learning rate scheduler or manually adjusting it for different epochs. A good practice is to start with a moderate learning rate and decrease it as training progresses.



2. Batch Size Adjustment

The batch size determines how many samples are processed before the model’s weights are updated. Smaller batch sizes can lead to noisy updates, whereas larger batch sizes make the model's training more stable but require more memory.

In practice, try experimenting with different batch sizes to find an optimal one that balances memory constraints and training stability.

3. Data Augmentation

Data augmentation helps in making the model robust by artificially increasing the size and variability of the training dataset. For object detection tasks, common augmentations include random flipping, scaling, rotation, and color jittering.

In addition to the RandomHorizontalFlip() transformation, you can also try other augmentations, such as:

    Random Vertical Flip: Flipping the image vertically can help the model recognize objects in both orientations.
    Color Jitter: Varying the brightness, contrast, saturation, or hue can help improve the model’s robustness to lighting conditions.
    Random Scaling/Resizing: Varying the size of the object within the image.

4. Model Architecture Tuning

    Backbone Selection: Faster R-CNN uses a backbone network (like ResNet50 or ResNet101). You can experiment with different backbones to see if they offer better accuracy at the cost of computational efficiency. For example, ResNet101 might provide more accuracy but may be slower to train, while MobileNetV2 can be faster but may not perform as well on more complex tasks.
    Anchor Generation: Faster R-CNN relies on predefined anchor boxes for object detection. Tuning the anchor generation mechanism (like changing anchor sizes and aspect ratios) can improve detection accuracy. However, this is a more advanced modification and may require further research.

5. Early Stopping

Using early stopping can help avoid overfitting and reduce training time. This technique stops training if the validation loss doesn’t improve after a set number of epochs.

6. Hyperparameter Tuning

You can also perform grid search or random search to tune other hyperparameters like:

    Momentum and Weight Decay: Try different values of momentum and weight decay for the optimizer.
    Number of Epochs: Training for more epochs can sometimes help, but you need to monitor for overfitting.
    ROI Pooling and Other Faster R-CNN Parameters: Fine-tune the Region of Interest (ROI) pooling and other related parameters to optimize performance further.

7. Transfer Learning

Fine-tuning a pre-trained model (like the one provided by torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)) allows you to leverage weights learned on a large dataset like ImageNet. However, you can also freeze some layers (like the backbone) to speed up training and reduce overfitting, or unfreeze the layers of interest for more granular tuning.